# Predicting Valorant Esports Map Outcomes
### Model 1: Logistic Regression
**DS 4400** | Rohil Singh, Sean Tian | TA: Matthew Laws

## Step 1 - Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

print('All imports loaded.')

## Step 2 - Load Data
Upload `VLRDataDummies.csv` and `VLRDataDropped.csv` to Colab.

- **Dummies** (127,830 rows): Missing values filled with 0
- **Dropped** (107,864 rows): Rows with missing values removed

We train on both and compare.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df_dummies = pd.read_csv('VLRDataDummies.csv').drop(columns=['Unnamed: 0'])
df_dropped = pd.read_csv('VLRDataDropped.csv').drop(columns=['Unnamed: 0'])

print(f'Dummies: {df_dummies.shape[0]:,} records, {df_dummies.shape[1]} columns')
print(f'Dropped: {df_dropped.shape[0]:,} records, {df_dropped.shape[1]} columns')
print(f'\nColumns: {df_dummies.columns.tolist()}')
df_dummies.head()

## Step 3 - Train/Test Split and Scaling
Stratified 80/20 split to preserve class balance. StandardScaler is fit on training data only.

In [ ]:
# Using Dummies dataset
X_dum = df_dummies.drop(columns=['Team1 Win'])
y_dum = df_dummies['Team1 Win']

X_train_dum, X_test_dum, y_train_dum, y_test_dum = train_test_split(
    X_dum, y_dum, test_size=0.2, random_state=42, stratify=y_dum
)

scaler_dum = StandardScaler()
X_train_dum_s = scaler_dum.fit_transform(X_train_dum)
X_test_dum_s = scaler_dum.transform(X_test_dum)

print(f'Dummies - Train: {X_train_dum.shape[0]:,}, Test: {X_test_dum.shape[0]:,}')

In [ ]:
# Using Dropped dataset
X_drop = df_dropped.drop(columns=['Team1 Win'])
y_drop = df_dropped['Team1 Win']

X_train_drop, X_test_drop, y_train_drop, y_test_drop = train_test_split(
    X_drop, y_drop, test_size=0.2, random_state=42, stratify=y_drop
)

scaler_drop = StandardScaler()
X_train_drop_s = scaler_drop.fit_transform(X_train_drop)
X_test_drop_s = scaler_drop.transform(X_test_drop)

print(f'Dropped - Train: {X_train_drop.shape[0]:,}, Test: {X_test_drop.shape[0]:,}')

---
# Logistic Regression
Logistic Regression learns a weight for each feature, computes a weighted sum, passes it through a sigmoid function, and outputs a probability between 0 and 1. If the probability is above 0.5, we predict Team 1 wins.

In [ ]:
def evaluate_lr(X_train_s, X_test_s, y_train, y_test, dataset_name):
    """Train and evaluate Logistic Regression, return model and results."""
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_s, y_train)

    pred = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:, 1]

    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)

    cv_scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring='accuracy')

    print(f'=== Logistic Regression Results ({dataset_name}) ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-Score:  {f1:.4f}')
    print(f'ROC-AUC:   {auc:.4f}')
    print(f'\n5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
    print(f'Individual folds: {[f"{s:.4f}" for s in cv_scores]}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, pred, target_names=['Loss', 'Win']))

    return model, pred, proba, {'Accuracy': acc, 'Precision': prec, 'Recall': rec,
                                 'F1': f1, 'ROC-AUC': auc, 'CV Mean': cv_scores.mean()}

print('Helper function defined.')

### Dummies Dataset (NaN filled with 0)

In [ ]:
lr_dum, pred_dum, proba_dum, results_dum = evaluate_lr(
    X_train_dum_s, X_test_dum_s, y_train_dum, y_test_dum, 'Dummies'
)

### Dropped Dataset (NaN rows removed)

In [ ]:
lr_drop, pred_drop, proba_drop, results_drop = evaluate_lr(
    X_train_drop_s, X_test_drop_s, y_train_drop, y_test_drop, 'Dropped'
)

### Side-by-Side Comparison

In [ ]:
comparison = pd.DataFrame({'Dummies': results_dum, 'Dropped': results_drop})
print(comparison.to_string())

### Log Loss Comparison
Log loss measures how confident and correct the model's probability predictions are. Lower is better. We compare training loss, test loss, and random predictions to confirm the model is actually learning.

In [ ]:
from sklearn.metrics import log_loss

for model, X_tr, X_te, y_tr, y_te, name in [
    (lr_dum, X_train_dum_s, X_test_dum_s, y_train_dum, y_test_dum, 'Dummies'),
    (lr_drop, X_train_drop_s, X_test_drop_s, y_train_drop, y_test_drop, 'Dropped')]:

    train_proba = model.predict_proba(X_tr)
    test_proba = model.predict_proba(X_te)
    random_proba = np.random.rand(len(y_te))

    print(f'=== {name} ===')
    print(f'  Training log loss: {log_loss(y_tr, train_proba):.4f}')
    print(f'  Test log loss:     {log_loss(y_te, test_proba):.4f}')
    print(f'  Random log loss:   {log_loss(y_te, random_proba):.4f}')
    print()

### Feature Coefficients
Which features does Logistic Regression rely on most? Positive coefficients help Team 1 win, negative coefficients hurt.

In [ ]:
feature_names = X_dum.columns.tolist()

print('=== Dummies ===')
coefs_dum = pd.DataFrame(
    zip(feature_names, np.transpose(lr_dum.coef_[0])),
    columns=['Feature', 'Coefficient']
).sort_values('Coefficient', key=abs, ascending=False)
print(f'Intercept: {lr_dum.intercept_[0]:.4f}')
print(coefs_dum.to_string(index=False))

print('\n=== Dropped ===')
coefs_drop = pd.DataFrame(
    zip(feature_names, np.transpose(lr_drop.coef_[0])),
    columns=['Feature', 'Coefficient']
).sort_values('Coefficient', key=abs, ascending=False)
print(f'Intercept: {lr_drop.intercept_[0]:.4f}')
print(coefs_drop.to_string(index=False))

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (pred, y_test, name) in zip(axes, [(pred_dum, y_test_dum, 'Dummies'),
                                            (pred_drop, y_test_drop, 'Dropped')]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Loss', 'Win'], yticklabels=['Loss', 'Win'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix ({name})')

plt.tight_layout()
plt.show()

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for proba, y_test, name, color in [(proba_dum, y_test_dum, 'Dummies', '#5B8FB9'),
                                    (proba_drop, y_test_drop, 'Dropped', '#E07A5F')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline (AUC = 0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - Logistic Regression')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

---
## Summary
Both datasets produce strong results with Logistic Regression. The close match between test accuracy and 5-fold CV accuracy indicates the model is not overfitting.